<a href="https://colab.research.google.com/github/nikitask14/neural-networks-pytorch-from-first-principles/blob/main/03_validation_and_classification/10_mlp_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [2]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(2,4)
    self.layer2 = nn.Linear(4,1)


  def forward(self,x):
    x = self.layer1(x)
    x = torch.relu(x)

    x = self.layer2(x)

    return x

# A tensor is PyTorch’s fundamental container for numerical data.
x_train = torch.tensor([
[1.0, 2.0],
[2.0, 1.0],
[1.5, 1.0],
[4.0, 5.0],
[5.0, 4.0],
[4.5, 4.0]
], dtype = torch.float32)

y_train = torch.tensor([
  [0],
  [0],
  [0],
  [1],
  [1],
  [1]
], dtype = torch.float32)

x_val = torch.tensor([
[1.0, 1.5],
[5.0, 5.0],
], dtype = torch.float32)

y_val = torch.tensor([
  [0],
  [1]
], dtype = torch.float32)

print(x_train.shape)
print(y_train.shape)
print(x_val.shape)
print(y_val.shape)

# MyModel is the architectural blueprint.
# model is the actual neural-network object created from that blueprint.
model = MyModel()

# This means Binary Cross Entropy With Logits Loss.
# Its job is to compare the model’s raw logits with the true binary
# labels and return a scalar loss value.
loss_fn = nn.BCEWithLogitsLoss()

# The optimiser needs to know which parameters it is allowed to update.
optimiser = torch.optim.SGD(model.parameters(), lr = 0.01)


# In our current full-batch setup,
# one epoch means that the model processes the entire training set
# once before moving to the next epoch
for epoch in range(100):

  # Put the model into training mode.
  model.train()

  # PyTorch accumulates gradients by default.
  # If we want each epoch to calculate a fresh gradient
  # based on the current loss, we clear the previous gradients first.
  optimiser.zero_grad()

  # Calling: model(x_train) causes PyTorch to execute the forward()
  # method defined in MyModel.
  train_logit = model(x_train)

  # starts at the loss and works backward through the computation graph
  # to calculate gradients for all learnable parameters.
  # This is backpropogation.
  train_loss = loss_fn(train_logit, y_train)
  train_loss.backward()

  # uses the gradients calculated by backward()
  # to update the model’s weights and biases
  optimiser.step()

  #Put the model into evaluation mode.
  model.eval()

  # For the operations inside this block,
  # do not build a computation graph for gradient calculation.
  with torch.no_grad():
    val_logit = model(x_val)
    val_loss = loss_fn(val_logit, y_val)

  if epoch % 10 == 0:
    print(f"Epoch:{epoch},"
    f"Training Loss:{train_loss.item()},"
    f"Validation Loss:{val_loss.item()}")

probability = torch.sigmoid(val_logit)
prediction = (probability >= 0.5).float()

correct = (prediction == y_val).float()
accuracy = correct.mean()
print("Predictions:", prediction)
print("Validation Labels:",y_val)
print("Accuracy", accuracy)


print("Validation logits:", val_logit)
print("Validation probabilities:", probability)


torch.Size([6, 2])
torch.Size([6, 1])
torch.Size([2, 2])
torch.Size([2, 1])
Epoch:0,Training Loss:0.6886637210845947,Validation Loss:0.7383279800415039
Epoch:10,Training Loss:0.6673552989959717,Validation Loss:0.7149611115455627
Epoch:20,Training Loss:0.6546321511268616,Validation Loss:0.6972085237503052
Epoch:30,Training Loss:0.6429233551025391,Validation Loss:0.6810780763626099
Epoch:40,Training Loss:0.6317736506462097,Validation Loss:0.6792811155319214
Epoch:50,Training Loss:0.6208824515342712,Validation Loss:0.6781629323959351
Epoch:60,Training Loss:0.6136613488197327,Validation Loss:0.6773437261581421
Epoch:70,Training Loss:0.6106656193733215,Validation Loss:0.6770310401916504
Epoch:80,Training Loss:0.6079119443893433,Validation Loss:0.6761113405227661
Epoch:90,Training Loss:0.6048572659492493,Validation Loss:0.6757428050041199
Predictions: tensor([[1.],
        [1.]])
Validation Labels: tensor([[0.],
        [1.]])
Accuracy tensor(0.5000)
Validation logits: tensor([[0.1006],
    